In [1]:
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client_llm = OpenAI(
    api_key=os.getenv('GROQ_TOKEN'),
    base_url=os.getenv('BASE_URL_GROQ')
)

client_db = Pinecone(api_key=os.getenv('PINE_API'))

emb_model = OpenAI(
    api_key=os.getenv('LOCAL_TOKEN'),
    base_url=os.getenv('LOCAL_URL')
)


In [2]:
articles = [
    {"id": "art-01", "headline": "SpaceX Launches New Starship Rocket",
     "topic": "Science", "year": 2024,
     "keywords": ["space", "rocket", "nasa", "exploration"]},
    {"id": "art-02", "headline": "Bitcoin Reaches All-Time High as Investors Rush In",
     "topic": "Business", "year": 2024,
     "keywords": ["crypto", "bitcoin", "finance", "investment"]},
    {"id": "art-03", "headline": "Champions League Final: Real Madrid vs Manchester City",
     "topic": "Sport", "year": 2023,
     "keywords": ["football", "soccer", "champions league", "madrid"]},
    {"id": "art-04", "headline": "New AI Model Outperforms Humans in Medical Diagnosis",
     "topic": "Tech", "year": 2024,
     "keywords": ["ai", "healthcare", "machine learning", "diagnosis"]},
    {"id": "art-05", "headline": "Global Leaders Meet to Discuss Climate Change",
     "topic": "Science", "year": 2023,
     "keywords": ["climate", "environment", "policy", "emissions"]},
    {"id": "art-06", "headline": "Apple Announces Revolutionary New iPhone",
     "topic": "Tech", "year": 2024,
     "keywords": ["apple", "iphone", "smartphone", "technology"]},
    {"id": "art-07", "headline": "Stock Markets Plunge Amid Recession Fears",
     "topic": "Business", "year": 2023,
     "keywords": ["stocks", "economy", "recession", "wall street"]},
    {"id": "art-08", "headline": "Olympics 2024: USA Wins Gold in Swimming",
     "topic": "Sport", "year": 2024,
     "keywords": ["olympics", "swimming", "usa", "gold medal"]},
    {"id": "art-09", "headline": "Scientists Discover New Species in Amazon Rainforest",
     "topic": "Science", "year": 2023,
     "keywords": ["biology", "amazon", "species", "nature"]},
    {"id": "art-10", "headline": "Tesla Unveils Autonomous Robot for Home Use",
     "topic": "Tech", "year": 2024,
     "keywords": ["tesla", "robot", "automation", "ai"]},
]

In [5]:
def create_article_text(article):
    return f'''Headline: {article['headline']}
Topic: {article['topic']}
Keywords: {', '.join(article['keywords'])}'''

def create_emdebbing(texts):
    emb = emb_model.embeddings.create(
        model=os.getenv('LOCAL_MODEL'),
        input=texts
    )

    emb_dict = emb.model_dump()

    return [data['embedding'] for data in emb_dict['data']]

client_db.create_index(
    name='manipulation-index',
    dimension=1024,
    spec=ServerlessSpec(
        cloud='aws',
        region='us-east-1'
    )
)

index = client_db.Index('manipulation-index')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
article_texts = [create_article_text(article) for article in articles]
article_embeddings = create_emdebbing(article_texts)


vectors = [
    {
        'id':article['id'],
        'values':article_embeddings[i],
        'metadata':{'topic':article['topic'], 'year':article['year']}
    }
    for i, article in enumerate(articles)
]

index.upsert(
    vectors=vectors
)

print(f'Данные загружены. Векторов: {len(vectors)}')

Данные загружены. Векторов: 10


In [ ]:
# Часть 1
## Задача 1.1

result = index.fetch(
    ids=['art-01','art-04']
)

In [40]:
for vector_id, vector_record in result.vectors.items():
    print(f'{vector_id} | {', '.join([f'{i}={v}' for i,v in vector_record['metadata'].items()])} | vector[:5]={[round(v,4) for v in vector_record['values'][:5]]}')
print(f'Read Units: {result.usage['read_units']}')

art-04 | topic=Tech, year=2024 | vector[:5]=[-0.0611, -0.0146, -0.0003, -0.0148, -0.0276]
art-01 | topic=Science, year=2024 | vector[:5]=[0.0187, -0.0248, -0.0037, -0.0202, -0.0191]
Read Units: 1


In [ ]:
## Задача 1.2

result = index.fetch(
    ids=['art-01','art-04'],
    namespace='nonexistent-ns'
)

print(result)
# Так как при добавление записей в индекс мы не указывали namespace-ы то следовательно все записи попали в namespace default что также можно заметить если просмотреть describe_index_stats()

print(index.describe_index_stats())
# В котором указан namespace '__default__': {количество записей}.

FetchResponse(namespace='nonexistent-ns', vectors={}, usage=None, _response_info={'raw_headers': {'date': 'Thu, 19 Mar 2026 05:40:21 GMT', 'content-type': 'application/json', 'content-length': '43', 'connection': 'keep-alive', 'x-pinecone-request-latency-ms': '38', 'x-envoy-upstream-service-time': '38', 'x-pinecone-response-duration-ms': '39', 'grpc-status': '0', 'server': 'envoy'}})
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '184',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 19 Mar 2026 05:40:21 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '63',
                                    'x-pinecone-request-latency-ms': '62',
                                    'x-pinecone-response-duration-ms': '64'}},

In [48]:
# Часть 2
## Задача 2.1

query = 'space exploration and rockets'
query_emb = create_emdebbing(query)

result = index.query(
    vector=query_emb,
    top_k=3
)

In [49]:
print(f'Запрос: {query}')
for i, match in enumerate(result.matches):
    print(f'    {i+1}. {match['id']} | score={round(match['score'], 4)}')

Запрос: space exploration and rockets
    1. art-01 | score=0.6311
    2. art-10 | score=0.4678
    3. art-04 | score=0.4013


In [51]:
## Задача 2.2

query = 'space exploration and rockets'
query_emb = create_emdebbing(query)

result = index.query(
    vector=query_emb,
    top_k=3,
    include_values=True
)

top_1 = result.matches[0]

print(f'ID: {top_1['id']}')
print(f'Score: {round(top_1['score'],4)}')
print(f'Vector[:5]: {[round(v,4) for v in top_1['values'][:5]]}')

ID: art-01
Score: 0.6311
Vector[:5]: [0.0187, -0.0248, -0.0037, -0.0202, -0.0191]


In [53]:
## Задача 2.3

client_db.delete_index('dotproduct-index')
client_db.create_index(
    name='dotproduct-index',
    dimension=1024,
    metric='dotproduct',
    spec=ServerlessSpec(
        cloud='aws',
        region='us-east-1'
    )
)

index_dot = client_db.Index('dotproduct-index')

In [55]:
vectors = [
    {
        'id':article['id'],
        'values':article_embeddings[i],
        'metadata':{'topic':article['topic'], 'year':article['year']}
    }
    for i, article in enumerate(articles)
]

index_dot.upsert(
    vectors=vectors
)

UpsertResponse(upserted_count=10, _response_info={'raw_headers': {'date': 'Thu, 19 Mar 2026 05:53:11 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '41399', 'x-pinecone-request-latency-ms': '1274', 'x-envoy-upstream-service-time': '248', 'x-pinecone-response-duration-ms': '1278', 'grpc-status': '0', 'server': 'envoy'}})

In [56]:
result_cosine = index.query(
    vector=query_emb,
    top_k=3
)

result_dot = index_dot.query(
    vector=query_emb,
    top_k=3
)

In [58]:
cosine_matches = result_cosine.matches
dot_matches = result_dot.matches

system_prompt = 'Ты senior ai engineer. Тебе передают результаты query метода к index-ам Pinecone, один индекс используется cosine расчет схожестви, а второй dotproduct. Сравни показатели результатов и объясни почему score двух разных подходов нельзя сравнивать напрямую.'

resposne = client_llm.chat.completions.create(
    model=os.getenv('GROQ_MODEL'),
    messages=[
        {
            'role':'system',
            'content':system_prompt
        },
        {
            'role':'user',
            'content':f'Cosine result: {cosine_matches}\nDotproduct result: {dot_matches}'
        }
    ]
)

print(resposne.choices[0].message.content)

<think>
Хорошо, давайте разберемся с этим вопросом. Итак, пользователь предоставляет результаты обработки запроса с использованием двух различных методов измерения схожести: косинусное сходство и скалярное произведение (dot product). Результаты в обоих случаях выглядят одинаково — id'ы документов такие же, и даже численные значения схожи до 9-го знака после запятой. Интересно, что в косинусном результате третий элемент имеет значение 0.401317626, а в dotproduct-е это 0.401317596. Разница небольшая, но она есть. 

Первое, что приходит в голову: что происходит с этими методами? Косинусное сходство измеряет направление векторов, а скалярное произведение учитывает и их длины. Значит, если векторы нормализованы, то скалярное произведение будет пропорционально косинусу угла между ними. Но в данном случае, видимо, векторы ненормализованы, и поэтому скалярное произведение может давать разные масштабы. Например, если векторы длинные, то скалярное произведение будет больше, даже если угол между 

In [65]:
# Часть 3
## Задача 3.1

queries = [
    'innovation and new technologies',
    'environmental issues'
]

queries_emb = create_emdebbing(queries)

print('=== Только Tech ===')
result = index.query(
    vector=queries_emb[0],
    top_k=3,
    filter={
        'topic':'Tech'
    },
    include_metadata=True
)

for i, match in enumerate(result.matches):
    print(f'  {i+1}. {match['id']} | score={match['score']:.4f} | {', '.join([f'{i}={v}' for i, v in match['metadata'].items()])}')

print('\n=== Только 2023 год ===')
result = index.query(
    vector=queries_emb[1],
    top_k=3,
    filter={
        'year':2023
    },
    include_metadata=True
)

for i, match in enumerate(result.matches):
    print(f'  {i+1}. {match['id']} | score={match['score']:.4f} | {', '.join([f'{i}={v}' for i, v in match['metadata'].items()])}')

=== Только Tech ===
  1. art-06 | score=0.5392 | topic=Tech, year=2024
  2. art-10 | score=0.5031 | topic=Tech, year=2024
  3. art-04 | score=0.5010 | topic=Tech, year=2024

=== Только 2023 год ===
  1. art-05 | score=0.4766 | topic=Science, year=2023
  2. art-09 | score=0.3747 | topic=Science, year=2023
  3. art-07 | score=0.2978 | topic=Business, year=2023


In [66]:
## Задача 3.2

queries = [
    'sports competitions',
    'financial markets'
]

queries_emb = create_emdebbing(queries)

print('=== год > 2023 ===')
result = index.query(
    vector=queries_emb[0],
    top_k=3,
    filter={
        'year':{'$gt':2023}
    },
    include_metadata=True
)

for i, match in enumerate(result.matches):
    print(f'  {i+1}. {match['id']} | score={match['score']:.4f} | {', '.join([f'{i}={v}' for i, v in match['metadata'].items()])}')

print('\n=== Исключить Business ===')
result = index.query(
    vector=queries_emb[1],
    top_k=3,
    filter={
        'topic':{'$ne':'Business'}
    },
    include_metadata=True
)

for i, match in enumerate(result.matches):
    print(f'  {i+1}. {match['id']} | score={match['score']:.4f} | {', '.join([f'{i}={v}' for i, v in match['metadata'].items()])}')

=== год > 2023 ===
  1. art-08 | score=0.5659 | topic=Sport, year=2024
  2. art-01 | score=0.3418 | topic=Science, year=2024
  3. art-04 | score=0.3289 | topic=Tech, year=2024

=== Исключить Business ===
  1. art-04 | score=0.3030 | topic=Tech, year=2024
  2. art-08 | score=0.2938 | topic=Sport, year=2024
  3. art-09 | score=0.2832 | topic=Science, year=2023


In [67]:
## Задача 3.3

query = 'breakthrough discoveries'
query_emb = create_emdebbing(query)

result = index.query(
    vector=query_emb,
    top_k=5,
    filter={
        'topic':'Science',
        'year':2023
    },
    include_metadata=True
)

print('=== Science И 2023 год ===')
for match in result.matches:
    print(f'  {match['id']} | {', '.join([f'{i}={v}' for i, v in match['metadata'].items()])}')

=== Science И 2023 год ===
  art-09 | topic=Science, year=2023
  art-05 | topic=Science, year=2023


In [68]:
# Часть 4
## Задача 4.1

result = index.fetch(
    ids=['art-07']
)

print(result)

FetchResponse(namespace='', vectors={'art-07': Vector(id='art-07', values=[-0.0551656112, 0.0419314317, -0.00692923088, 0.0474173762, 0.0495896488, -0.0503050797, -0.0475662649, -0.0134524554, -0.0925548896, -0.0620140247, 0.00287801214, -0.0119409496, 0.052124586, -0.00477766711, -0.0329223722, 0.0764336139, -0.0333462134, -0.056632448, 0.0275238752, -0.0356166, 0.0129941702, -0.0673166588, -0.0243252181, 0.0497480333, -0.0220894106, -0.0721278414, -0.0368359163, 0.0265364647, -0.0148337837, -0.0789980665, 0.0426402204, -0.0184661411, -0.0344244577, -0.0184623711, 0.0135924481, -0.0100748194, 0.000457460497, 0.0284027345, 0.0378013626, 0.0336315669, 0.0383077301, 0.0559834056, -0.0142009873, 0.0221081935, 0.0440823324, -0.0112863425, 0.0287019219, -0.0100227054, 0.0253413096, 0.0499465689, -0.0209396631, 0.0209436491, -0.0146765606, 0.0402044952, -0.0336035229, -0.0211260412, -0.0203821473, -0.00609546294, -0.00638844492, -0.0186131895, -0.000753769884, 0.0748831853, -0.0285427682, -0

In [73]:
new_text = '''Headline: Stock Markets Rally Following Fed Rate Cut
Topic: Business
Keywords: stocks, economy, federal reserve, recovery'''

new_text_emb = create_emdebbing(new_text)[0]

index.update(
    id='art-07',
    values=new_text_emb
)

new_result = index.fetch(ids=['art-07'])
print(new_result)

FetchResponse(namespace='', vectors={'art-07': Vector(id='art-07', values=[-0.0475500934, 0.00513299042, -0.00817763247, 0.0208555069, 0.0561267398, -0.0349884778, -0.0593898371, -0.0146453502, -0.0497880615, 0.0101900315, -0.0170087274, 0.0545151532, 0.051145941, -0.00473133475, -0.0458353311, 0.0627094433, -0.115989439, -0.0268502943, 0.00247870339, 0.0015543045, -0.00208153971, -0.0867508352, -0.00983231794, 0.0762169957, 0.0248660278, -0.0260195732, -0.00310946396, 0.028013492, -0.0107133463, -0.0618533939, 0.0178725012, 0.0418372415, -0.054467231, -0.00245935284, 0.012681108, -0.0110594258, -0.00491474429, 0.0261971466, 0.00238701259, 0.0342445076, 0.0349606648, 0.0757159442, -0.0231456663, 0.00332519, 0.0255675074, -0.00881202426, 0.0388370492, -0.00495704357, 0.0424155369, 0.0616972521, -0.0106177684, 0.0130296452, -0.0114089698, 0.0186387375, 0.00034648148, 0.0129404739, -0.00890966691, 0.0394531861, -0.0133298589, -0.0199256148, 0.0106409593, 0.0880351737, -0.00997889508, -0.0

In [74]:
print(f'''До обновления: vector[:5]={[round(v, 4) for v in result.vectors['art-07']['values'][:5]]}
После обновления: vector[:5]={[round(v, 4) for v in new_result.vectors['art-07']['values'][:5]]} -> Значения изменились''')

До обновления: vector[:5]=[-0.0552, 0.0419, -0.0069, 0.0474, 0.0496]
После обновления: vector[:5]=[-0.0476, 0.0051, -0.0082, 0.0209, 0.0561] -> Значения изменились


In [82]:
## Задача 4.2

index.update(
    id='art-07',
    set_metadata={'year':2024, 'rating':4}
)

updated_result = index.fetch(
    ids=['art-07'],
)

In [84]:
updated_metadata = updated_result.vectors['art-07']['metadata']
print(f'''topic: {updated_metadata['topic']}
year: {updated_metadata['year']}
rating: {updated_metadata['rating']}''')

topic: Business
year: 2024
rating: 4


In [85]:
## Задача 4.3

new_query_emb = create_emdebbing('Football final match 2024')[0]

index.update(
    id='art-03',
    values=new_query_emb,
    set_metadata={'year':2024, 'rating':5}
)

result = index.fetch(ids=['art-03'])

In [87]:
result.vectors['art-03']['metadata']

{'rating': 5, 'topic': 'Sport', 'year': 2024}

In [89]:
# Часть 5
## Задача 5.1

before_delete = index.describe_index_stats()['total_vector_count']

index.delete(ids=['art-09','art-10'])

after_delete = index.describe_index_stats()['total_vector_count']

result = index.fetch(ids=['art-09','art-10'])

In [ ]:
print(f'''До удаления: {before_delete}
После удаления: {after_delete}
fetch удалённой записи: {result.vectors}''')
# Pinecone возвращает пустой словарь так как не удалось найти указанные иднетификаторы векторов

До удаления: 10
После удаления: 8
fetch удалённой записи: {}


In [91]:
## Задача 5.2

before_delete = index.describe_index_stats()['total_vector_count']
index.delete(filter={'topic':'Sport'})
result = index.fetch(ids=['art-03','art-08'])
after_delete = index.describe_index_stats()['total_vector_count']

In [92]:
print(f'''До удаления: {before_delete}
После удаления: {after_delete}
fetch удалённой записи: {result.vectors}''')
# Pinecone возвращает пустой словарь так как не удалось найти указанные иднетификаторы векторов

До удаления: 8
После удаления: 6
fetch удалённой записи: {}


In [97]:
# Задача 6

archive_2023_vectors = [vector for article, vector in zip(articles, vectors) if article['year']==2023]
live_2024_vectors = [vector for article, vector in zip(articles, vectors) if article['year']==2024]

index.upsert(
    vectors=archive_2023_vectors,
    namespace='archive-2023'
)

index.upsert(
    vectors=live_2024_vectors,
    namespace='live-2024'
)

print(f'Namespaces: {index.describe_index_stats()['namespaces']}')



Namespaces: {'__default__': {'vector_count': 6}, 'archive-2023': {'vector_count': 4}, 'live-2024': {'vector_count': 6}}


In [99]:
query_emb = create_emdebbing('ai technology')
result = index.query(
    vector=query_emb,
    top_k=3,
    namespace='live-2024',
    include_metadata=True
)

for match in result.matches:
    print(match)

{'id': 'art-04',
 'metadata': {'topic': 'Tech', 'year': 2024},
 'score': 0.555517256,
 'values': []}
{'id': 'art-10',
 'metadata': {'topic': 'Tech', 'year': 2024},
 'score': 0.50173378,
 'values': []}
{'id': 'art-06',
 'metadata': {'topic': 'Tech', 'year': 2024},
 'score': 0.395420074,
 'values': []}


In [100]:
index.delete(delete_all=True, namespace='archive-2023')
print(index.describe_index_stats()['namespaces'])

{'live-2024': {'vector_count': 6}, '__default__': {'vector_count': 6}}


In [101]:
client_db.delete_index('manipulation-index')
client_db.list_indexes()

[
    {
        "name": "dotproduct-index",
        "metric": "dotproduct",
        "host": "dotproduct-index-uhdd6kj.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1024,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "datacamp-index",
        "metric": "cosine",
        "host": "datacamp-index-uhdd6kj.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "